# Experiment 5: Triple Extraction from Structured Data

**Goal:** Convert MusicBrainz + Discogs JSON into RDF triples using `rdflib`. Output Turtle format.

**Key questions:**
- What does the mapping from JSON fields to RDF triples look like?
- How do we define our ontology namespace and use existing ones (MO, CIDOC-CRM)?
- What does the generated Turtle output look like?
- How do we handle release deduplication?

## Setup

In [1]:
import musicbrainzngs
import requests
import json
import time
from rdflib import Graph, Namespace, Literal, URIRef, BNode
from rdflib.namespace import RDF, RDFS, XSD, FOAF, OWL

# API setup
musicbrainzngs.set_useragent("KE-CW2-MusicHistory", "0.1", "lucas@example.com")
HEADERS = {"User-Agent": "KE-CW2-MusicHistory/0.1 (lucas@example.com)"}
DISCOGS_BASE = "https://api.discogs.com"

def discogs_get(endpoint, params=None):
    url = f"{DISCOGS_BASE}{endpoint}"
    resp = requests.get(url, headers=HEADERS, params=params)
    if resp.status_code == 429:
        time.sleep(60)
        resp = requests.get(url, headers=HEADERS, params=params)
    resp.raise_for_status()
    return resp.json()

# Define namespaces
MO = Namespace("http://purl.org/ontology/mo/")
CRM = Namespace("http://www.cidoc-crm.org/cidoc-crm/")
MH = Namespace("http://example.org/music-history/")  # Our custom namespace
MB = Namespace("http://musicbrainz.org/artist/")
MB_RELEASE = Namespace("http://musicbrainz.org/release/")
MB_RECORDING = Namespace("http://musicbrainz.org/recording/")
DC = Namespace("http://purl.org/dc/elements/1.1/")
SCHEMA = Namespace("https://schema.org/")

# Create the graph
g = Graph()
g.bind("mo", MO)
g.bind("crm", CRM)
g.bind("mh", MH)
g.bind("mb", MB)
g.bind("mb_release", MB_RELEASE)
g.bind("mb_recording", MB_RECORDING)
g.bind("foaf", FOAF)
g.bind("dc", DC)
g.bind("schema", SCHEMA)

print("Graph created with namespaces bound.")
print(f"Namespaces: mo, crm, mh, mb, foaf, dc, schema")

Graph created with namespaces bound.
Namespaces: mo, crm, mh, mb, foaf, dc, schema


## 1. Map a Single Artist (MusicBrainz → RDF)

Let's manually map David Bowie's MusicBrainz data to triples, seeing exactly what each mapping produces.

In [2]:
# Fetch Bowie from MusicBrainz
mbid = "5441c29d-3602-4898-b1a1-b77fa23b8e50"
mb_detail = musicbrainzngs.get_artist_by_id(
    mbid, includes=["tags", "url-rels", "artist-rels"]
)["artist"]

print(f"Fetched: {mb_detail['name']} ({mbid})")

Fetched: David Bowie (5441c29d-3602-4898-b1a1-b77fa23b8e50)


In [3]:
# --- Map basic artist info ---
artist_uri = MB[mbid]

# Type: Person → mo:SoloMusicArtist, Group → mo:MusicGroup
artist_type = mb_detail.get("type", "")
if artist_type == "Person":
    g.add((artist_uri, RDF.type, MO.SoloMusicArtist))
elif artist_type == "Group":
    g.add((artist_uri, RDF.type, MO.MusicGroup))
else:
    g.add((artist_uri, RDF.type, MO.MusicArtist))

# Name
g.add((artist_uri, FOAF.name, Literal(mb_detail["name"])))
g.add((artist_uri, RDFS.label, Literal(mb_detail["name"])))

# Country
country = mb_detail.get("country")
if country:
    g.add((artist_uri, MH.countryOfOrigin, Literal(country)))

# Birth/death dates
life_span = mb_detail.get("life-span", {})
if life_span.get("begin"):
    g.add((artist_uri, SCHEMA.birthDate, Literal(life_span["begin"], datatype=XSD.date)))
if life_span.get("end"):
    g.add((artist_uri, SCHEMA.deathDate, Literal(life_span["end"], datatype=XSD.date)))

# Gender
gender = mb_detail.get("gender")
if gender:
    g.add((artist_uri, SCHEMA.gender, Literal(gender)))

# Birth place
begin_area = mb_detail.get("begin-area", {})
if begin_area.get("name"):
    g.add((artist_uri, MH.birthPlace, Literal(begin_area["name"])))

print(f"Added basic info triples for {mb_detail['name']}")
print(f"Graph now has {len(g)} triples")

Added basic info triples for David Bowie
Graph now has 8 triples


In [4]:
# --- Map genres/tags ---
tags = mb_detail.get("tag-list", [])
genre_count = 0

for tag in tags:
    count = int(tag.get("count", 0))
    if count >= 3:  # Filter low-confidence tags
        genre_name = tag["name"]
        # Create a URI for the genre
        genre_uri = MH[f"genre/{genre_name.replace(' ', '_').lower()}"]
        g.add((genre_uri, RDF.type, MO.Genre))
        g.add((genre_uri, RDFS.label, Literal(genre_name)))
        g.add((artist_uri, MO.genre, genre_uri))
        genre_count += 1

print(f"Added {genre_count} genre triples (filtered by count >= 3)")
print(f"Graph now has {len(g)} triples")

Added 12 genre triples (filtered by count >= 3)
Graph now has 44 triples


In [5]:
# --- Map artist relationships ---
artist_rels = mb_detail.get("artist-relation-list", [])
rel_count = 0

for rel in artist_rels:
    rel_type = rel.get("type", "")
    target = rel.get("artist", {})
    target_mbid = target.get("id")
    target_name = target.get("name", "Unknown")
    
    if not target_mbid:
        continue
    
    target_uri = MB[target_mbid]
    # Ensure target has a label
    g.add((target_uri, RDFS.label, Literal(target_name)))
    
    if rel_type == "member of band":
        direction = rel.get("direction", "")
        if direction == "forward":  # artist is member of target band
            g.add((artist_uri, MO.member_of, target_uri))
            g.add((target_uri, RDF.type, MO.MusicGroup))
        else:  # target is member of artist (artist is the band)
            g.add((target_uri, MO.member_of, artist_uri))
        
        # Instruments from attributes
        for attr in rel.get("attribute-list", []):
            if attr not in ["original", "minor"]:
                instrument_uri = MH[f"instrument/{attr.replace(' ', '_').lower()}"]
                g.add((instrument_uri, RDF.type, MO.Instrument))
                g.add((instrument_uri, RDFS.label, Literal(attr)))
                # Link the member to the instrument
                if direction == "forward":
                    g.add((artist_uri, MH.playsInstrument, instrument_uri))
                else:
                    g.add((target_uri, MH.playsInstrument, instrument_uri))
        rel_count += 1
    
    elif rel_type == "collaboration":
        g.add((artist_uri, MH.collaboratedWith, target_uri))
        rel_count += 1
    
    elif rel_type == "influenced by":
        g.add((artist_uri, MH.influencedBy, target_uri))
        rel_count += 1

print(f"Added {rel_count} relationship triples")
print(f"Graph now has {len(g)} triples")

Added 18 relationship triples
Graph now has 148 triples


## 2. Map Releases (Albums + Tracks)

Use release groups to avoid duplication.

In [6]:
# Fetch release groups (deduplicated albums)
time.sleep(1.1)
release_groups = musicbrainzngs.browse_release_groups(
    artist=mbid,
    release_type=["album"],
    limit=5  # Just first 5 for testing
)

print(f"Release groups found: {release_groups.get('release-group-count', '?')}")
print(f"Fetching first {len(release_groups['release-group-list'])}:\n")

for rg in release_groups["release-group-list"]:
    rg_id = rg["id"]
    title = rg["title"]
    rg_type = rg.get("type", "N/A")
    first_release = rg.get("first-release-date", "N/A")
    
    # Create album URI and triples
    album_uri = MH[f"album/{rg_id}"]
    g.add((album_uri, RDF.type, MO.Release))
    g.add((album_uri, RDFS.label, Literal(title)))
    g.add((album_uri, DC.title, Literal(title)))
    g.add((artist_uri, MH.released, album_uri))
    
    if first_release and first_release != "N/A":
        g.add((album_uri, MH.releaseDate, Literal(first_release)))
    
    print(f"  {title} ({first_release}) → {album_uri}")

print(f"\nGraph now has {len(g)} triples")

Release groups found: 419
Fetching first 5:

  David Bowie (1967-06) → http://example.org/music-history/album/5db40351-97ed-36a8-88e2-a21a2603cae1
  David Bowie (1969-11-04) → http://example.org/music-history/album/2e12918c-4973-3537-b9ab-e4723ae1ae1d
  The Man Who Sold the World (1970-11-04) → http://example.org/music-history/album/2536a41d-fde9-35d5-a6c6-cd4d94ffd916
  Hunky Dory (1971-12-17) → http://example.org/music-history/album/743b0b2e-a23a-3182-950e-232f8cb0dfb7
  The Rise and Fall of Ziggy Stardust and the Spiders From Mars (1972-06-06) → http://example.org/music-history/album/6c9ae3dd-32ad-472c-96be-69d0a3536261

Graph now has 173 triples


In [7]:
# Fetch tracks for the first album
time.sleep(1.1)
first_rg = release_groups["release-group-list"][0]
first_rg_id = first_rg["id"]

# Get releases in this release group
rg_releases = musicbrainzngs.browse_releases(
    release_group=first_rg_id, limit=1
)

if rg_releases["release-list"]:
    release_id = rg_releases["release-list"][0]["id"]
    time.sleep(1.1)
    release_detail = musicbrainzngs.get_release_by_id(
        release_id, includes=["recordings"]
    )["release"]
    
    album_uri = MH[f"album/{first_rg_id}"]
    print(f"Tracks for: {release_detail['title']}\n")
    
    for medium in release_detail.get("medium-list", []):
        for track in medium.get("track-list", []):
            rec = track.get("recording", {})
            rec_id = rec.get("id")
            rec_title = rec.get("title", "Unknown")
            
            if rec_id:
                track_uri = MB_RECORDING[rec_id]
                g.add((track_uri, RDF.type, MO.Track))
                g.add((track_uri, DC.title, Literal(rec_title)))
                g.add((track_uri, RDFS.label, Literal(rec_title)))
                g.add((album_uri, MH.hasTrack, track_uri))
                
                # Duration
                length = rec.get("length")
                if length:
                    g.add((track_uri, MH.duration, Literal(int(length), datatype=XSD.integer)))
                
                print(f"  {track.get('position', '?')}. {rec_title} ({rec_id})")

print(f"\nGraph now has {len(g)} triples")

Tracks for: David Bowie

  1. Uncle Arthur (87d8cbad-f34e-4162-a809-a38fd49248f8)
  2. Sell Me a Coat (008fbf08-649f-4257-b6cc-78cb6d441c22)
  3. Rubber Band (3f0b540a-fed2-4ec8-b8dd-e2d19c660eee)
  4. Love You Till Tuesday (24b01ae9-c5ae-434a-b5b4-bfbd7af04cb4)
  5. There Is a Happy Land (a45052d7-d5f0-4b60-afdd-9a72a8755507)
  6. We Are Hungry Men (2a851410-fb7e-439c-82f1-439cb98606e7)
  7. When I Live My Dream (725a4d31-5e4a-413d-80a7-eb40c6d1e8a4)
  8. Little Bombardier (56838c0b-5fbb-4b8c-a041-e6fcee78c75e)
  9. Silly Boy Blue (2e3ae1f9-3f3d-4480-b6ec-65ed14455f80)
  10. Come and Buy My Toys (f2c28327-2f24-4108-9ad0-5c3fa3cb9cfa)
  11. Join the Gang (ffc068b1-8b43-44c4-a891-725ad5b5267d)
  12. She’s Got Medals (0e5fca0a-93be-4295-8546-d6c9f063a8ed)
  13. Maid of Bond Street (ca638ace-b2f0-4437-bd2a-74e70d1a40f5)
  14. Please Mr. Gravedigger (09fae20a-a5b8-4263-a49b-aa9d7d48dde3)

Graph now has 243 triples


## 3. Add Discogs Data (Genres/Styles)

Enrich with Discogs genre → style hierarchy.

In [8]:
# Fetch from Discogs
time.sleep(3)
dc_artist = discogs_get("/artists/10263")

# Add real name
realname = dc_artist.get("realname")
if realname:
    g.add((artist_uri, MH.realName, Literal(realname)))

# Fetch a few releases for genre/style data
time.sleep(3)
dc_releases = discogs_get(f"/artists/10263/releases", params={"per_page": 5})

all_genres = set()
all_styles = set()

for rel in dc_releases["releases"][:3]:
    rel_id = rel["id"]
    rel_type = rel.get("type", "release")
    time.sleep(3)
    
    try:
        if rel_type == "master":
            detail = discogs_get(f"/masters/{rel_id}")
        else:
            detail = discogs_get(f"/releases/{rel_id}")
        
        genres = detail.get("genres", [])
        styles = detail.get("styles", [])
        all_genres.update(genres)
        all_styles.update(styles)
        
        # Create genre → style (subgenre) triples
        for genre in genres:
            genre_uri = MH[f"genre/{genre.replace(' ', '_').replace('/', '_').lower()}"]
            g.add((genre_uri, RDF.type, MO.Genre))
            g.add((genre_uri, RDFS.label, Literal(genre)))
            
            for style in styles:
                style_uri = MH[f"genre/{style.replace(' ', '_').replace('/', '_').lower()}"]
                g.add((style_uri, RDF.type, MO.Genre))
                g.add((style_uri, RDFS.label, Literal(style)))
                g.add((style_uri, MH.subgenreOf, genre_uri))
        
        print(f"  {detail.get('title', '?'):40s} | {genres} → {styles}")
    except Exception as e:
        print(f"  Error: {e}")

print(f"\nGenres: {sorted(all_genres)}")
print(f"Styles: {sorted(all_styles)}")
print(f"Graph now has {len(g)} triples")

  I Do Believe I Love You                  | ['Rock', 'Pop'] → ['Vocal']
  I Live In Dreams                         | ['Rock', 'Pop'] → ['Vocal', 'Acoustic']
  London Boys                              | ['Rock', 'Pop'] → ['Mod']

Genres: ['Pop', 'Rock']
Styles: ['Acoustic', 'Mod', 'Vocal']
Graph now has 258 triples


## 4. View the Generated Turtle

Let's see what our RDF looks like.

In [9]:
# Serialize to Turtle and display
turtle_output = g.serialize(format="turtle")
print(turtle_output)

@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix mb: <http://musicbrainz.org/artist/> .
@prefix mb_recording: <http://musicbrainz.org/recording/> .
@prefix mh: <http://example.org/music-history/> .
@prefix mo: <http://purl.org/ontology/mo/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema: <https://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://example.org/music-history/genre/acoustic> a mo:Genre ;
    rdfs:label "Acoustic" ;
    mh:subgenreOf <http://example.org/music-history/genre/pop>,
        <http://example.org/music-history/genre/rock> .

<http://example.org/music-history/genre/mod> a mo:Genre ;
    rdfs:label "Mod" ;
    mh:subgenreOf <http://example.org/music-history/genre/pop>,
        <http://example.org/music-history/genre/rock> .

<http://example.org/music-history/genre/vocal> a mo:Genre ;
    rdfs:label "Vocal" ;
    mh:subgenreOf <http://example.org/music-history/genre

## 5. Save to File

In [10]:
output_path = "../ontology/experiment5_sample.ttl"
g.serialize(destination=output_path, format="turtle")
print(f"Saved to {output_path}")
print(f"Total triples: {len(g)}")

Saved to ../ontology/experiment5_sample.ttl
Total triples: 258


## 6. Test SPARQL Query Against the Graph

Let's query our own graph to verify it works.

In [11]:
# CQ1: What genres does David Bowie belong to?
query1 = """
PREFIX mo: <http://purl.org/ontology/mo/>
PREFIX mh: <http://example.org/music-history/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?genreName WHERE {
    ?artist rdfs:label "David Bowie" .
    ?artist mo:genre ?genre .
    ?genre rdfs:label ?genreName .
}
"""

print("CQ: What genres does David Bowie belong to?\n")
for row in g.query(query1):
    print(f"  - {row.genreName}")

CQ: What genres does David Bowie belong to?

  - actors
  - alternative rock
  - art pop
  - art rock
  - british
  - classic rock
  - glam rock
  - new wave
  - pop
  - Pop
  - pop rock
  - rock
  - Rock
  - uk


In [12]:
# CQ: What bands was David Bowie a member of?
query2 = """
PREFIX mo: <http://purl.org/ontology/mo/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?bandName WHERE {
    ?artist rdfs:label "David Bowie" .
    ?artist mo:member_of ?band .
    ?band rdfs:label ?bandName .
}
"""

print("CQ: What bands was David Bowie a member of?\n")
for row in g.query(query2):
    print(f"  - {row.bandName}")

CQ: What bands was David Bowie a member of?

  - David Bowie & The Buzz
  - The Arnold Corns
  - The Manish Boys
  - Bewlay Bros.
  - Davy Jones & The Lower Third
  - Tin Machine
  - Feathers
  - The Konrads
  - The Riot Squad
  - David Bowie & The Hype
  - The Hype
  - Tao Jones Index
  - The Astronettes
  - Davie Jones & The King Bees


In [13]:
# CQ: What tracks are on the first album?
query3 = """
PREFIX mh: <http://example.org/music-history/>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?albumTitle ?trackTitle WHERE {
    ?artist rdfs:label "David Bowie" .
    ?artist mh:released ?album .
    ?album mh:hasTrack ?track .
    ?album dc:title ?albumTitle .
    ?track dc:title ?trackTitle .
}
ORDER BY ?albumTitle ?trackTitle
"""

print("CQ: What tracks are on albums?\n")
current_album = None
for row in g.query(query3):
    if row.albumTitle != current_album:
        current_album = row.albumTitle
        print(f"\n  Album: {current_album}")
    print(f"    - {row.trackTitle}")

CQ: What tracks are on albums?


  Album: David Bowie
    - Come and Buy My Toys
    - Join the Gang
    - Little Bombardier
    - Love You Till Tuesday
    - Maid of Bond Street
    - Please Mr. Gravedigger
    - Rubber Band
    - Sell Me a Coat
    - She’s Got Medals
    - Silly Boy Blue
    - There Is a Happy Land
    - Uncle Arthur
    - We Are Hungry Men
    - When I Live My Dream


In [14]:
# CQ: What subgenres exist and what are their parent genres?
query4 = """
PREFIX mh: <http://example.org/music-history/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?styleName ?genreName WHERE {
    ?style mh:subgenreOf ?genre .
    ?style rdfs:label ?styleName .
    ?genre rdfs:label ?genreName .
}
ORDER BY ?genreName ?styleName
"""

print("CQ: What subgenres exist?\n")
for row in g.query(query4):
    print(f"  {row.styleName} → subgenreOf → {row.genreName}")

CQ: What subgenres exist?

  Acoustic → subgenreOf → Pop
  Mod → subgenreOf → Pop
  Vocal → subgenreOf → Pop
  Acoustic → subgenreOf → Rock
  Mod → subgenreOf → Rock
  Vocal → subgenreOf → Rock
  Acoustic → subgenreOf → pop
  Mod → subgenreOf → pop
  Vocal → subgenreOf → pop
  Acoustic → subgenreOf → rock
  Mod → subgenreOf → rock
  Vocal → subgenreOf → rock


## 7. Summary

| Aspect | Observation |
|---|---|
| Mapping approach | rdflib in Python, manual field→triple mapping |
| Ontology used | Music Ontology (mo:) for types, custom (mh:) for properties |
| Triple count | Fill in after running |
| Turtle output | Clean? Readable? |
| SPARQL queries | Do CQs work against the graph? |
| Release deduplication | Using release groups instead of raw releases |
| Genre hierarchy | Discogs genres→styles mapped to mh:subgenreOf |

### Mapping Rules Applied

| Source Field | → | RDF Triple |
|---|---|---|
| MB `type=Person` | → | `(artist, rdf:type, mo:SoloMusicArtist)` |
| MB `type=Group` | → | `(artist, rdf:type, mo:MusicGroup)` |
| MB `name` | → | `(artist, foaf:name, "name")` |
| MB `country` | → | `(artist, mh:countryOfOrigin, "GB")` |
| MB `life-span.begin` | → | `(artist, schema:birthDate, "1947-01-08"^^xsd:date)` |
| MB `tag-list` (count≥3) | → | `(artist, mo:genre, genre_uri)` |
| MB `rel: member of band` | → | `(artist, mo:member_of, band_uri)` |
| MB `rel: collaboration` | → | `(artist, mh:collaboratedWith, other_uri)` |
| MB release group | → | `(artist, mh:released, album_uri)` |
| MB recording | → | `(album, mh:hasTrack, track_uri)` |
| Discogs `realname` | → | `(artist, mh:realName, "David Robert Jones")` |
| Discogs `genres/styles` | → | `(style, mh:subgenreOf, genre)` |

## Next Steps

- **Experiment 6**: Extract triples from Wikipedia text using NER/LLM
- **Scale up**: Turn this mapping into a reusable function that processes any artist
- **Add ontology declarations**: Define our custom classes and properties formally in the .ttl